In [ ]:
import numpy as np
import lal
import taichi as ti

ti.init(
    arch=ti.cpu,
    default_fp=ti.f64,
    cpu_max_num_threads=1,
    offline_cache=False,
    debug=True,
)
from matplotlib import pyplot as plt

%matplotlib inline

from pespace.detector.orbit import (
    KaplerianHeliocentric,
    KeplerianGeocentric,
    ConstellationVectorStruct,
)

In [ ]:
tsamples_np = np.arange(0, lal.YRJUL_SI, 3600)
tsamples = ti.field(ti.f64, shape=tsamples_np.shape)
tsamples.from_numpy(tsamples_np)

orb_helio = KaplerianHeliocentric(2.5e9, 0.0, 0.0)
orb_geo = KeplerianGeocentric(1.0e8, 0.0, 0.0)

orb_vec_container_helio = ConstellationVectorStruct.field(shape=tsamples_np.shape)
orb_vec_container_geo = ConstellationVectorStruct.field(shape=tsamples_np.shape)

AU_SEC = lal.AU_SI / lal.C_SI

In [ ]:
@ti.kernel
def orbit_test():
    for i in tsamples:
        orb_vec_container_helio[i] = orb_helio.get_constellation_vectors(tsamples[i])
        orb_vec_container_geo[i] = orb_geo.get_constellation_vectors(tsamples[i])


orbit_test()

In [ ]:
data_helio = orb_vec_container_helio.to_numpy()
data_geo = orb_vec_container_geo.to_numpy()

## KaplerianHeliocentric orbit model

In [ ]:
diff_n3 = (data_helio["x2"] - data_helio["x1"]) / orb_helio.arm_length_sec - data_helio["n3"]
diff_n2 = (data_helio["x1"] - data_helio["x3"]) / orb_helio.arm_length_sec - data_helio["n2"]
diff_n1 = (data_helio["x3"] - data_helio["x2"]) / orb_helio.arm_length_sec - data_helio["n1"]
print(np.linalg.norm(diff_n1, axis=1).max())
print(np.linalg.norm(diff_n2, axis=1).max())
print(np.linalg.norm(diff_n3, axis=1).max())

In [ ]:
plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, data_helio["x1"][:, 0] / AU_SEC, label="node 1")
plt.plot(tsamples_np, data_helio["x2"][:, 0] / AU_SEC, label="node 2")
plt.plot(tsamples_np, data_helio["x3"][:, 0] / AU_SEC, label="node 3")
plt.ylabel("x (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, data_helio["x1"][:, 1] / AU_SEC, label="node 1")
plt.plot(tsamples_np, data_helio["x2"][:, 1] / AU_SEC, label="node 2")
plt.plot(tsamples_np, data_helio["x3"][:, 1] / AU_SEC, label="node 3")
plt.ylabel("y (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, data_helio["x1"][:, 2] / orb_helio.arm_length_sec, label="node 1")
plt.plot(tsamples_np, data_helio["x2"][:, 2] / orb_helio.arm_length_sec, label="node 2")
plt.plot(tsamples_np, data_helio["x3"][:, 2] / orb_helio.arm_length_sec, label="node 3")
plt.ylabel("z (Larm)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
det_center_helio = np.array(
    [
        AU_SEC * np.cos(2 * lal.PI / lal.YRJUL_SI * tsamples_np),
        AU_SEC * np.sin(2 * lal.PI / lal.YRJUL_SI * tsamples_np),
        np.zeros(tsamples_np.shape),
    ]
).T
data_helio["x1_det"] = data_helio["x1"] - det_center_helio
data_helio["x2_det"] = data_helio["x2"] - det_center_helio
data_helio["x3_det"] = data_helio["x3"] - det_center_helio

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, (data_helio["x1_det"][:, 0]) / orb_helio.arm_length_sec, label="node 1")
plt.plot(tsamples_np, (data_helio["x2_det"][:, 0]) / orb_helio.arm_length_sec, label="node 2")
plt.plot(tsamples_np, (data_helio["x3_det"][:, 0]) / orb_helio.arm_length_sec, label="node 3")
plt.ylabel("x_det (Larm)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, (data_helio["x1_det"][:, 1]) / orb_helio.arm_length_sec, label="node 1")
plt.plot(tsamples_np, (data_helio["x2_det"][:, 1]) / orb_helio.arm_length_sec, label="node 2")
plt.plot(tsamples_np, (data_helio["x3_det"][:, 1]) / orb_helio.arm_length_sec, label="node 3")
plt.ylabel("y_det (Larm)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
center = (data_helio["x1"] + data_helio["x2"] + data_helio["x3"]) / 3

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 0] / AU_SEC, label="center (sum)")
plt.plot(tsamples_np, det_center_helio[:, 0] / AU_SEC, label="center (analytic)")
plt.ylabel("x (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 1] / AU_SEC, label="center (sum)")
plt.plot(tsamples_np, det_center_helio[:, 1] / AU_SEC, label="center (analytic)")
plt.ylabel("y (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 2] / AU_SEC, label="center (sum)")
plt.plot(tsamples_np, det_center_helio[:, 2] / AU_SEC, label="center (analytic)")
plt.ylabel("z (AU)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
# normal of the constellation plane
norm_vec = np.cross(data_helio["n1"], data_helio["n2"])
norm_vec = norm_vec / np.linalg.norm(norm_vec, axis=1, keepdims=True)

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, norm_vec[:, 0], label="norm vec x")
plt.plot(tsamples_np, norm_vec[:, 1], label="norm vec y")
plt.plot(tsamples_np, norm_vec[:, 2], label="norm vec z")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(
    tsamples_np,
    np.cos(2 * np.pi * tsamples_np / lal.YRJUL_SI + np.pi) * np.cos(np.pi / 6),
    label="norm vec x",
)
plt.plot(
    tsamples_np,
    np.sin(2 * np.pi * tsamples_np / lal.YRJUL_SI + np.pi) * np.cos(np.pi / 6),
    label="norm vec y",
)
plt.plot(
    tsamples_np, np.ones(tsamples_np.shape) * np.sin(np.pi / 6), label="norm vec z"
)
plt.xlabel("time (sec.)")
plt.legend()

## Comparing with bbhx

In [ ]:
from lisatools.detector import EqualArmlengthOrbits

orbit_bbhx = EqualArmlengthOrbits()
orbit_bbhx.configure(linear_interp_setup=True)

pos_x1_bbhx = orbit_bbhx.get_pos(tsamples_np, 1)
pos_x2_bbhx = orbit_bbhx.get_pos(tsamples_np, 2)
pos_x3_bbhx = orbit_bbhx.get_pos(tsamples_np, 3)

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, pos_x1_bbhx[:, 0] / lal.AU_SI, label="node 1")
plt.plot(tsamples_np, pos_x2_bbhx[:, 0] / lal.AU_SI, label="node 2")
plt.plot(tsamples_np, pos_x3_bbhx[:, 0] / lal.AU_SI, label="node 3")
plt.ylabel("x (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, pos_x1_bbhx[:, 1] / lal.AU_SI, label="node 1")
plt.plot(tsamples_np, pos_x2_bbhx[:, 1] / lal.AU_SI, label="node 2")
plt.plot(tsamples_np, pos_x3_bbhx[:, 1] / lal.AU_SI, label="node 3")
plt.ylabel("y (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, pos_x1_bbhx[:, 2] / orbit_bbhx.armlength, label="node 1")
plt.plot(tsamples_np, pos_x2_bbhx[:, 2] / orbit_bbhx.armlength, label="node 2")
plt.plot(tsamples_np, pos_x3_bbhx[:, 2] / orbit_bbhx.armlength, label="node 3")
plt.ylabel("z (Larm)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
det_center_helio = np.array(
    [
        lal.AU_SI * np.cos(2 * lal.PI / lal.YRJUL_SI * tsamples_np),
        lal.AU_SI * np.sin(2 * lal.PI / lal.YRJUL_SI * tsamples_np),
        np.zeros(tsamples_np.shape),
    ]
).T

plt.figure(figsize=[8, 2])
plt.plot(
    tsamples_np,
    (pos_x1_bbhx - det_center_helio)[:, 0] / orbit_bbhx.armlength,
    label="node 1",
)
plt.plot(
    tsamples_np,
    (pos_x2_bbhx - det_center_helio)[:, 0] / orbit_bbhx.armlength,
    label="node 2",
)
plt.plot(
    tsamples_np,
    (pos_x3_bbhx - det_center_helio)[:, 0] / orbit_bbhx.armlength,
    label="node 3",
)
plt.ylabel("x_det (Larm)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(
    tsamples_np,
    (pos_x1_bbhx - det_center_helio)[:, 1] / orbit_bbhx.armlength,
    label="node 1",
)
plt.plot(
    tsamples_np,
    (pos_x2_bbhx - det_center_helio)[:, 1] / orbit_bbhx.armlength,
    label="node 2",
)
plt.plot(
    tsamples_np,
    (pos_x3_bbhx - det_center_helio)[:, 1] / orbit_bbhx.armlength,
    label="node 3",
)
plt.ylabel("y_det (Larm)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
center = (pos_x1_bbhx + pos_x2_bbhx + pos_x3_bbhx) / 3

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 0] / AU_SEC, label="center bbhx (sum)")
plt.plot(tsamples_np, det_center_helio[:, 0] / AU_SEC, label="center bbhx (analytic)")
plt.ylabel("x (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 1] / AU_SEC, label="center bbhx (sum)")
plt.plot(tsamples_np, det_center_helio[:, 1] / AU_SEC, label="center bbhx (analytic)")
plt.ylabel("y (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 2] / AU_SEC, label="center bbhx (sum)")
plt.plot(tsamples_np, det_center_helio[:, 2] / AU_SEC, label="center bbhx (analytic)")
plt.ylabel("z (AU)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
# LINKS = [12, 23, 31, 13, 32, 21]
n12_bbhx = orbit_bbhx.get_normal_unit_vec(tsamples_np, 12)
n21_bbhx = orbit_bbhx.get_normal_unit_vec(tsamples_np, 21)
n23_bbhx = orbit_bbhx.get_normal_unit_vec(tsamples_np, 23)
n32_bbhx = orbit_bbhx.get_normal_unit_vec(tsamples_np, 32)
n13_bbhx = orbit_bbhx.get_normal_unit_vec(tsamples_np, 13)
n31_bbhx = orbit_bbhx.get_normal_unit_vec(tsamples_np, 31)
x1x2_bbhx = pos_x1_bbhx - pos_x2_bbhx
x1x2_bbhx = x1x2_bbhx / np.linalg.norm(x1x2_bbhx, axis=1, keepdims=True)
x2x1_bbhx = pos_x2_bbhx - pos_x1_bbhx
x2x1_bbhx = x2x1_bbhx / np.linalg.norm(x2x1_bbhx, axis=1, keepdims=True)
x2x3_bbhx = pos_x2_bbhx - pos_x3_bbhx
x2x3_bbhx = x2x3_bbhx / np.linalg.norm(x2x3_bbhx, axis=1, keepdims=True)
x3x2_bbhx = pos_x3_bbhx - pos_x2_bbhx
x3x2_bbhx = x3x2_bbhx / np.linalg.norm(x3x2_bbhx, axis=1, keepdims=True)
x3x1_bbhx = pos_x3_bbhx - pos_x1_bbhx
x3x1_bbhx = x3x1_bbhx / np.linalg.norm(x3x1_bbhx, axis=1, keepdims=True)
x1x3_bbhx = pos_x1_bbhx - pos_x3_bbhx
x1x3_bbhx = x1x3_bbhx / np.linalg.norm(x1x3_bbhx, axis=1, keepdims=True)

fig, (ax_x, ax_y, ax_z) = plt.subplots(3, 1, sharex=True, figsize=[8, 6])
plt.subplots_adjust(hspace=0)
ax_x.plot(tsamples_np, n12_bbhx[:, 0], label="n12")
ax_x.plot(tsamples_np, x1x2_bbhx[:, 0], label="x1-x2")
ax_x.set_ylabel("x")
ax_x.set_xlabel("time (sec.)")
ax_x.legend()
ax_y.plot(tsamples_np, n12_bbhx[:, 1], label="n12")
ax_y.plot(tsamples_np, x1x2_bbhx[:, 1], label="x1-x2")
ax_y.set_ylabel("y")
ax_y.set_xlabel("time (sec.)")
ax_y.legend()
ax_z.plot(tsamples_np, n12_bbhx[:, 2], label="n12")
ax_z.plot(tsamples_np, x1x2_bbhx[:, 2], label="x1-x2")
ax_z.set_ylabel("z")
ax_z.set_xlabel("time (sec.)")
ax_z.legend()

fig, (ax_x, ax_y, ax_z) = plt.subplots(3, 1, sharex=True, figsize=[8, 6])
plt.subplots_adjust(hspace=0)
ax_x.plot(tsamples_np, n21_bbhx[:, 0], label="n21")
ax_x.plot(tsamples_np, x2x1_bbhx[:, 0], label="x2-x1")
ax_x.set_ylabel("x")
ax_x.set_xlabel("time (sec.)")
ax_x.legend()
ax_y.plot(tsamples_np, n21_bbhx[:, 1], label="n21")
ax_y.plot(tsamples_np, x2x1_bbhx[:, 1], label="x2-x1")
ax_y.set_ylabel("y")
ax_y.set_xlabel("time (sec.)")
ax_y.legend()
ax_z.plot(tsamples_np, n21_bbhx[:, 2], label="n21")
ax_z.plot(tsamples_np, x2x1_bbhx[:, 2], label="x2-x1")
ax_z.set_ylabel("z")
ax_z.set_xlabel("time (sec.)")
ax_z.legend()

print(np.linalg.norm((n12_bbhx - x1x2_bbhx), axis=1).max())
print(np.linalg.norm((n21_bbhx - x2x1_bbhx), axis=1).max())
print(np.linalg.norm((n23_bbhx - x2x3_bbhx), axis=1).max())
print(np.linalg.norm((n32_bbhx - x3x2_bbhx), axis=1).max())
print(np.linalg.norm((n31_bbhx - x3x1_bbhx), axis=1).max())
print(np.linalg.norm((n13_bbhx - x1x3_bbhx), axis=1).max())

## KeplerianGeocentric orbit model 

In [ ]:
diff_n3 = (data_geo["x2"] - data_geo["x1"]) / orb_geo.arm_length_sec - data_geo["n3"]
diff_n2 = (data_geo["x1"] - data_geo["x3"]) / orb_geo.arm_length_sec - data_geo["n2"]
diff_n1 = (data_geo["x3"] - data_geo["x2"]) / orb_geo.arm_length_sec - data_geo["n1"]
print(np.linalg.norm(diff_n1, axis=1).max())
print(np.linalg.norm(diff_n2, axis=1).max())
print(np.linalg.norm(diff_n3, axis=1).max())

In [ ]:
plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, data_geo["x1"][:, 0] / AU_SEC, label="node 1")
plt.plot(tsamples_np, data_geo["x2"][:, 0] / AU_SEC, label="node 2")
plt.plot(tsamples_np, data_geo["x3"][:, 0] / AU_SEC, label="node 3")
plt.ylabel("x (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, data_geo["x1"][:, 1] / AU_SEC, label="node 1")
plt.plot(tsamples_np, data_geo["x2"][:, 1] / AU_SEC, label="node 2")
plt.plot(tsamples_np, data_geo["x3"][:, 1] / AU_SEC, label="node 3")
plt.ylabel("y (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, data_geo["x1"][:, 2] / orb_geo.arm_length_sec, label="node 1")
plt.plot(tsamples_np, data_geo["x2"][:, 2] / orb_geo.arm_length_sec, label="node 2")
plt.plot(tsamples_np, data_geo["x3"][:, 2] / orb_geo.arm_length_sec, label="node 3")
plt.ylabel("z (Larm)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, data_geo["x1"][:, 2] / orb_geo.arm_length_sec, label="node 1")
plt.plot(tsamples_np, data_geo["x2"][:, 2] / orb_geo.arm_length_sec, label="node 2")
plt.plot(tsamples_np, data_geo["x3"][:, 2] / orb_geo.arm_length_sec, label="node 3")
plt.ylabel("z (Larm)")
plt.xlim(0, 7 * lal.DAYJUL_SI)
plt.xticks(np.arange(7) * lal.DAYJUL_SI, np.arange(7))
plt.xlabel("time (day)")
plt.legend()

In [ ]:
det_center = np.array(
    [
        AU_SEC * np.cos(2 * lal.PI / lal.YRJUL_SI * tsamples_np),
        AU_SEC * np.sin(2 * lal.PI / lal.YRJUL_SI * tsamples_np),
        np.zeros(tsamples_np.shape),
    ]
).T
data_geo["x1_det"] = data_geo["x1"] - det_center
data_geo["x2_det"] = data_geo["x2"] - det_center
data_geo["x3_det"] = data_geo["x3"] - det_center

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, (data_geo["x1_det"][:, 0]) / orb_geo.arm_length_sec, label="node 1")
plt.plot(tsamples_np, (data_geo["x2_det"][:, 0]) / orb_geo.arm_length_sec, label="node 2")
plt.plot(tsamples_np, (data_geo["x3_det"][:, 0]) / orb_geo.arm_length_sec, label="node 3")
plt.ylabel("x_det (Larm)")
plt.xlim(0, 7 * lal.DAYJUL_SI)
plt.xticks(np.arange(7) * lal.DAYJUL_SI, np.arange(7))
plt.xlabel("time (day)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, (data_geo["x1_det"][:, 1]) / orb_geo.arm_length_sec, label="node 1")
plt.plot(tsamples_np, (data_geo["x2_det"][:, 1]) / orb_geo.arm_length_sec, label="node 2")
plt.plot(tsamples_np, (data_geo["x3_det"][:, 1]) / orb_geo.arm_length_sec, label="node 3")
plt.ylabel("y_det (Larm)")
plt.xlim(0, 7 * lal.DAYJUL_SI)
plt.xticks(np.arange(7) * lal.DAYJUL_SI, np.arange(7))
plt.xlabel("time (day)")
plt.legend()

In [ ]:
center = (data_geo["x1"] + data_geo["x2"] + data_geo["x3"]) / 3

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 0] / AU_SEC, label="center (sum)")
plt.plot(tsamples_np, det_center[:, 0] / AU_SEC, label="center (analytic)")
plt.ylabel("x (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 1] / AU_SEC, label="center (sum)")
plt.plot(tsamples_np, det_center[:, 1] / AU_SEC, label="center (analytic)")
plt.ylabel("y (AU)")
plt.xlabel("time (sec.)")
plt.legend()

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, center[:, 2] / AU_SEC, label="center (sum)")
plt.plot(tsamples_np, det_center[:, 2] / AU_SEC, label="center (analytic)")
plt.ylabel("z (AU)")
plt.xlabel("time (sec.)")
plt.legend()

In [ ]:
# normal of the constellation plane
norm_vec = np.cross(data_geo["n1"], data_geo["n2"])
norm_vec = norm_vec / np.linalg.norm(norm_vec, axis=1, keepdims=True)

plt.figure(figsize=[8, 2])
plt.plot(tsamples_np, norm_vec[:, 0], label="norm vec x")
plt.plot(tsamples_np, norm_vec[:, 1], label="norm vec y")
plt.plot(tsamples_np, norm_vec[:, 2], label="norm vec z")
plt.xlabel("time (sec.)")
plt.legend()

lam_ref = 120.5 / 180 * np.pi
beta_ref = -4.7 / 180 * np.pi
plt.figure(figsize=[8, 2])
plt.plot(
    tsamples_np,
    np.ones(tsamples_np.shape) * np.cos(lam_ref) * np.cos(beta_ref),
    label="norm vec x",
)
plt.plot(
    tsamples_np,
    np.ones(tsamples_np.shape) * np.sin(lam_ref) * np.cos(beta_ref),
    label="norm vec y",
)
plt.plot(
    tsamples_np, 
    np.ones(tsamples_np.shape) * np.sin(beta_ref), 
    label="norm vec z",
    )
plt.xlabel("time (sec.)")
plt.legend()